# 0.0 Bibliotecas Importadas

In [2]:
import pandas as pd

# 0.1 Carregando Dataset

In [3]:
df_raw = pd.read_csv('/mnt/c/Users/T.i/Documents/Repos/projeto_RH/dataset/HR-Employee-Attrition-balanced.csv')

# 0.2 Leitura do Dataset

In [4]:
df_raw.head(10).T

,0,1,2,3,4,5,6,7,8,9
Age,58,27,50,39,32,29,33,30,42,32
Attrition,Yes,Yes,Yes,No,No,Yes,No,Yes,No,Yes
BusinessTravel,Travel_Rarely,Travel_Frequently,Travel_Rarely,Non-Travel,Travel_Rarely,Travel_Frequently,Travel_Rarely,Travel_Frequently,Travel_Rarely,Travel_Frequently
DailyRate,601,1341,407,439,601,337,501,1441,557,825
Department,Research & Development,Human Resources,Sales,Research & Development,Sales,Research & Development,Research & Development,Research & Development,Research & Development,Research & Development
DistanceFromHome,7,21,28,9,7,14,15,0,18,29
Education,4,2,3,3,5,1,2,5,4,4
EducationField,Medical,Human Resources,Marketing,Life Sciences,Marketing,Other,Medical,Life Sciences,Life Sciences,Medical
EmployeeCount,1,1,1,1,1,1,1,1,1,1
EmployeeNumber,1360,1950,2049,1132,1446,1421,2009,1442,1998,1089


# 1.0 Baseline 

In [10]:
baseline_df = df_raw.copy()

baseline_df['Annual_Salary'] = baseline_df['MonthlyIncome']*12
baseline_df['Replacement_Multiplier'] = baseline_df['JobLevel'].apply(lambda x: 0.40 if x <= 2 else (1.00 if x == 3 else 2.00))
baseline_df['Replacement_Cost'] = baseline_df['Annual_Salary'] * baseline_df['Replacement_Multiplier']

#formando um df apenas os funcionários que tiveram a saída da empresa
leaving_df = baseline_df.loc[baseline_df['Attrition'] == 'Yes', :]

#metricas
total_employees = len(baseline_df)
employees_leaving = len(leaving_df)
attrition_rate = (employees_leaving / total_employees) * 100
avg_replacement_cost = leaving_df['Replacement_Cost'].mean()
total_annual_cost = leaving_df['Replacement_Cost'].sum()

print("="*60)
print("SITUAÇÃO ATUAL (BASELINE SEM MACHINE LEARNING)")
print("="*60)
print(f"Total de Funcionários: {total_employees}")
print(f"Funcionários que saem/ano (Rotatividade): {employees_leaving}")
print(f"Taxa de Rotatividade Anual: {attrition_rate:.2f}%")

print("-"*60)
print(f"Custo médio de substituição/pessoa: ${avg_replacement_cost:,.2f}")
print(f"Custo total anual de rotatividade: ${total_annual_cost:,.2f}")
print("="*60)

SITUAÇÃO ATUAL (BASELINE SEM MACHINE LEARNING)
Total de Funcionários: 1761
Funcionários que saem/ano (Rotatividade): 528
Taxa de Rotatividade Anual: 29.98%
------------------------------------------------------------
Custo médio de substituição/pessoa: $46,195.30
Custo total anual de rotatividade: $24,391,116.00


### 1.1 Simulação do ROI - Simples

In [ ]:
#o quanto o meu modelo performa
recall = 0.10

#o quanto o time de RH performa para trabalhar em cima dos empregados que o modelo acusou
retention_rate = 0.30

#custos:

#manutenção
maintenance_annual_cost = 2000

#custo RH para intervir
invtervetion_cost = 5000

#custo de produzir o modelo em si
production_cost = 20000

#pessoas que o algoritmo aponta como potencial de demissão
true_positive = int(employees_leaving * recall)

#desempenho do rh
retention_employee = int(true_positive * retention_rate)
avg_saving = retention_employee * avg_replacement_cost

#custo anual
annual_cost = maintenance_annual_cost + invtervetion_cost + production_cost

#resultado liquido
liquid_savings = avg_saving - annual_cost
roi = (avg_saving / annual_cost) * 100

print(f'A média de economia da retenção dos empregados foi de: R${avg_saving:.2f}')
print(f'O ROI foi de: R${roi:.2f}')




A média de economia da retenção dos empregados foi de: R$692929.43
O ROI foi de: R$2566.41


### 1.2 Simulação do ROI - Elaborado

In [19]:
def simular_cenario_roi(recall, precision, taxa_retencao):

    # Constantes do Report
    CUSTO_MANUTENCAO_ANUAL = 20000
    CUSTO_INTERVENCAO = 5000

    # 1. Pessoas que a Máquina Aponta (Recall)
    verdadeiros_positivos = int(employees_leaving * recall)

    # 2. Pessoas Retidas de Fato pelo RH
    funcionarios_retidos = int(verdadeiros_positivos * taxa_retencao)

    # 3. Benefício Bruto (Pessoas salvas x O quanto a empresa gastaria pra demiti-las)
    economia_bruta = funcionarios_retidos * avg_replacement_cost
    
    # 5. Custos do Processo (Precisamos pagar pela manutenção de TI e pelas reuniões do RH)
    # Quantos alarmes o modelo gerou no total para conseguirmos os Verdadeiros Positivos?
    # Se a precisão é 70%, o volume total alardado é Verdadeiros Positivos / 0.70
    total_intencoes_rh = verdadeiros_positivos / precision if precision > 0 else 0
    custo_intervencoes = total_intencoes_rh * CUSTO_INTERVENCAO
    
    custos_anuais_totais = CUSTO_MANUTENCAO_ANUAL + custo_intervencoes
    
    # 6. Balanço Final
    economia_liquida = economia_bruta - custos_anuais_totais
    roi_anual = (economia_liquida / custos_anuais_totais) * 100 if custos_anuais_totais > 0 else 0
    
    return {
        "Cenário": "Simulado",
        "Precision": f"{precision*100:.0f}%",
        "Recall": f"{recall*100:.0f}%",
        "Retenção": f"{taxa_retencao*100:.0f}%",
        "Func. Retidos": funcionarios_retidos,
        "Economia Líquida (Anual)": f"${economia_liquida:,.0f}",
        "ROI Anual (%)": f"{roi_anual:.0f}%"
    }

# Criando tabela interativa com cenários conservador, moderado e otimista
tabela_analise = [
    simular_cenario_roi(precision = 0.60, recall=0.50, taxa_retencao=0.20),
    simular_cenario_roi(precision = 0.70, recall=0.60, taxa_retencao=0.30),
    simular_cenario_roi(precision = 0.80, recall=0.70, taxa_retencao=0.40),
]

df_potencial_roi = pd.DataFrame(tabela_analise)

# Renomear índices para identificar os cenários
df_potencial_roi['Cenário'] = ['Conservador', 'Moderado', 'Otimista']

display(df_potencial_roi)

,Cenário,Precision,Recall,Retenção,Func. Retidos,Economia Líquida (Anual),ROI Anual (%)
0,Conservador,60%,50%,20%,52,"$182,155",8%
1,Moderado,70%,60%,30%,94,"$2,065,215",91%
2,Otimista,80%,70%,40%,147,"$4,464,458",192%
